# Writing records and annotations

`write_record` writes a header and a signal file. `write_annotations` writes
an annotation file. In this notebook, the files are written to the current
directory.

You must give the signal parameters: the sampling frequency, the gain and
ADC resolution of each signal, and the units of each calibrated signal.
`write_record` has no defaults for them.

In [ ]:
import numpy as np
import wfdbx

# Records are written to, and read from, the current directory (no input
# or output directory is set).

## A synthetic record

This example writes a record, `synth`, with two signals sampled at 500 Hz: a 1.2 Hz "pulse" in millivolts and a slow
respiration-like signal in millimeters of mercury.

In [ ]:
fs = 500
t = np.arange(10 * fs) / fs
pulse = 1.5 * np.sin(2 * np.pi * 1.2 * t) ** 63           # mV
resp = 5 * np.sin(2 * np.pi * 0.25 * t)                   # mmHg
x = np.column_stack([pulse, resp])

wfdbx.write_record("synth", x, fs=fs, physical=True,
                   gains=[1000, 100],        # ADC units per mV, per mmHg
                   adcres=16, units=["mV", "mmHg"],
                   names=["pulse", "resp"],
                   comments=["synthetic example record"])
print(open("synth.hea").read())

Reading it back gives the same values, up to the resolution of the ADC
units (1/1000 mV and 1/100 mmHg here):

In [ ]:
back = wfdbx.read_record("synth", physical=True)
print(back.names, back.units, back.fs)
print("largest difference:", np.abs(back.signals - x).max(axis=0))

## Missing parameters are errors

If a parameter is missing, `write_record` raises a `ValueError` that names
it:

In [ ]:
try:
    wfdbx.write_record("bad", x, fs=fs, physical=True, adcres=16,
                       units="mV")
except ValueError as err:
    print("ValueError:", err)

A gain of 0 marks a signal as *uncalibrated*. Its samples are written in
ADC units, and it has no physical units. This example writes a record,
`counts`, with one uncalibrated signal.

In [ ]:
counts = np.round(100 * np.sin(2 * np.pi * t)).astype(int)
wfdbx.write_record("counts", counts[:, None], fs=fs, gains=0, adcres=16)
print(open("counts.hea").read())

## Annotations

Give the annotation times (sample numbers) and their mnemonics. This
example writes the annotation file `synth.pk` for the record `synth` written
above:

In [ ]:
peaks = np.flatnonzero((pulse[1:-1] > pulse[:-2]) & (pulse[1:-1] >= pulse[2:])) + 1
wfdbx.write_annotations("synth", "pk", peaks, symbols=["N"] * len(peaks))
pk = wfdbx.read_annotations("synth", "pk")
print(len(pk), "annotations; first times (s):", pk.samples[:5] / pk.fs)